# 자율주제: LIN28A 결합 강도와 번역 변화의 관계 분석

## 주제
**LIN28A에 강하게 결합한 mRNA일수록 Lin28a knockdown 후 ribosome density가 증가하는가?**  
또한 그 유전자들이 **ER/membrane/secretory pathway 관련 단백질**에 치우쳐 있는지 확인한다.

## 논문 배경
Cho et al. 논문은 LIN28A CLIP-seq으로 LIN28A가 결합하는 RNA를 찾고, ribosome footprinting으로 Lin28a knockdown 후 번역 변화를 분석했다. 논문 결론은 LIN28A가 단순히 let-7 precursor만 조절하는 것이 아니라 많은 mRNA에 결합하며, 특히 ER-associated translation을 억제한다는 것이다.

## 이 노트북에서 하는 일
1. `read-counts.txt`에서 CLIP/RNA/RPF read count를 불러온다.
2. 유전자별 LIN28A 결합 강도(`CLIP enrichment`)를 계산한다.
3. Lin28a knockdown 후 번역 변화(`ribosome density change`)를 계산한다.
4. LIN28A 결합 상위 유전자군과 하위 유전자군의 번역 변화를 통계적으로 비교한다.
5. localization annotation을 붙여 ER/membrane 관련 유전자들이 더 강하게 영향을 받는지 확인한다.
6. 후보 direct target gene table을 만든다.

> 이전 주차 과제의 Figure 4D/5B 재현에서 한 단계 확장해서, **통계검정 + 후보 유전자 리스트 생성**까지 하는 자율주제입니다.


In [ ]:

# Google Colab에서 실행하는 경우 Drive를 마운트합니다.
# 로컬/서버에서 실행한다면 이 셀은 건너뛰어도 됩니다.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Colab이 아니거나 이미 마운트되어 있습니다:', e)


In [ ]:

from pathlib import Path
import os

# 이전 주차와 동일한 작업 폴더를 기본값으로 사용합니다.
WORKDIR = Path('/content/drive/MyDrive/binfo1-work')

# 로컬에서 실행하는 경우 현재 폴더를 사용합니다.
if not WORKDIR.exists():
    WORKDIR = Path.cwd()

os.chdir(WORKDIR)
print('Working directory:', Path.cwd())

# 결과 저장 폴더
Path('results').mkdir(exist_ok=True)
Path('figures').mkdir(exist_ok=True)


In [ ]:

# 필요한 Python 패키지 확인
import sys, subprocess, importlib.util

required = ['pandas', 'numpy', 'matplotlib', 'scipy']
for pkg in required:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, mannwhitneyu, kruskal


## 1. read-counts.txt 불러오기

In [ ]:

import pandas as pd
from pathlib import Path

count_file = Path('read-counts.txt')
if not count_file.exists():
    raise FileNotFoundError(
        'read-counts.txt가 현재 폴더에 없습니다. '\
        '이전 1주차 노트북에서 featureCounts를 실행했거나, '\
        'binfo1-work 폴더에 read-counts.txt를 복사했는지 확인하세요.'
    )

cnts = pd.read_csv(count_file, sep='\t', comment='#', index_col=0)
print(cnts.shape)
cnts.head()


In [ ]:

# 이 분석에 필요한 column이 있는지 확인합니다.
required_cols = [
    'CLIP-35L33G.bam',
    'RNA-control.bam',
    'RNA-siLuc.bam',
    'RNA-siLin28a.bam',
    'RPF-siLuc.bam',
    'RPF-siLin28a.bam'
]

missing = [c for c in required_cols if c not in cnts.columns]
if missing:
    raise ValueError(f'필요한 column이 없습니다: {missing}')

cnts[required_cols].describe()


## 2. CLIP enrichment와 ribosome density change 계산

In [ ]:

import numpy as np

# 0 count 때문에 생기는 문제를 줄이기 위해 pseudocount 1을 더합니다.
pseudo = 1

df = cnts.copy()

# read count가 너무 낮은 유전자는 비율이 불안정하므로 제외합니다.
# 기준은 너무 엄격하지 않게 RNA count 10 초과로 설정했습니다.
mask = (
    (df['RNA-control.bam'] > 10) &
    (df['RNA-siLuc.bam'] > 10) &
    (df['RNA-siLin28a.bam'] > 10)
)
df = df.loc[mask].copy()

# LIN28A CLIP enrichment: LIN28A CLIP read / input RNA read
# 값이 클수록 LIN28A가 해당 mRNA에 더 많이 결합했다고 해석합니다.
df['clip_enrichment'] = (df['CLIP-35L33G.bam'] + pseudo) / (df['RNA-control.bam'] + pseudo)
df['clip_log2'] = np.log2(df['clip_enrichment'])

# Ribosome density change after Lin28a knockdown
# RPF/RNA = mRNA abundance로 보정한 ribosome occupancy입니다.
df['rden_siLuc'] = (df['RPF-siLuc.bam'] + pseudo) / (df['RNA-siLuc.bam'] + pseudo)
df['rden_siLin28a'] = (df['RPF-siLin28a.bam'] + pseudo) / (df['RNA-siLin28a.bam'] + pseudo)
df['rden_change'] = df['rden_siLin28a'] / df['rden_siLuc']
df['rden_log2'] = np.log2(df['rden_change'])

# RNA abundance 자체가 변했는지도 같이 계산합니다.
# 번역 조절 후보를 고를 때 RNA 변화가 너무 큰 유전자는 제외할 수 있습니다.
df['rna_change'] = (df['RNA-siLin28a.bam'] + pseudo) / (df['RNA-siLuc.bam'] + pseudo)
df['rna_change_log2'] = np.log2(df['rna_change'])

# 무한대/결측 제거
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=['clip_log2', 'rden_log2', 'rna_change_log2'])

print('분석에 사용한 유전자 수:', len(df))
df[['clip_log2', 'rden_log2', 'rna_change_log2']].describe()


## 3. LIN28A 결합 강도와 번역 변화의 상관관계

In [ ]:

# Figure 4D를 확장한 scatter plot
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(df['clip_log2'], df['rden_log2'], s=6, alpha=0.25)

# 단순 추세선
x = df['clip_log2'].values
y = df['rden_log2'].values
coef = np.polyfit(x, y, 1)
xs = np.linspace(np.nanpercentile(x, 1), np.nanpercentile(x, 99), 200)
ax.plot(xs, coef[0] * xs + coef[1], linewidth=2)

ax.axhline(0, linewidth=1)
ax.axvline(0, linewidth=1)
ax.set_xlabel('log2(CLIP enrichment)')
ax.set_ylabel('log2(ribosome density change; siLin28a / siLuc)')
ax.set_title('LIN28A binding vs translation change')
plt.tight_layout()
plt.savefig('figures/fig1_clip_vs_rden_scatter.png', dpi=200)
plt.show()

rho, pval = spearmanr(df['clip_log2'], df['rden_log2'])
print(f'Spearman correlation rho = {rho:.4f}, p-value = {pval:.3e}')


## 4. Strong binder와 weak binder 비교

In [ ]:

# LIN28A 결합 강도에 따라 유전자를 그룹화합니다.
# 논문 Figure 4E처럼 strong binder와 weak binder를 비교하는 방식입니다.
q95 = df['clip_log2'].quantile(0.95)
q80 = df['clip_log2'].quantile(0.80)
q50 = df['clip_log2'].quantile(0.50)

def assign_clip_group(v):
    if v >= q95:
        return 'Top 5% binders'
    elif v >= q80:
        return 'Top 5-20% binders'
    elif v <= q50:
        return 'Bottom 50% binders'
    else:
        return 'Middle 30% binders'

df['clip_group'] = df['clip_log2'].apply(assign_clip_group)

order = ['Bottom 50% binders', 'Middle 30% binders', 'Top 5-20% binders', 'Top 5% binders']

# 그룹별 boxplot
plot_data = [df.loc[df['clip_group'] == g, 'rden_log2'] for g in order]
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(plot_data, labels=order, showfliers=False)
ax.axhline(0, linewidth=1)
ax.set_ylabel('log2(ribosome density change)')
ax.set_title('Translation change by LIN28A binding strength')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('figures/fig2_rden_by_clip_group_boxplot.png', dpi=200)
plt.show()

# 통계검정: Top 5% vs Bottom 50%
top = df.loc[df['clip_group'] == 'Top 5% binders', 'rden_log2']
bottom = df.loc[df['clip_group'] == 'Bottom 50% binders', 'rden_log2']
stat, p = mannwhitneyu(top, bottom, alternative='two-sided')
print(f'Top 5% vs Bottom 50% Mann-Whitney U p-value = {p:.3e}')
print(f'Top 5% median rden_log2 = {top.median():.4f}')
print(f'Bottom 50% median rden_log2 = {bottom.median():.4f}')


## 5. ER/membrane/secretory 관련 유전자 분석

In [ ]:

# Localization annotation을 불러옵니다.
# 이전 과제에서 사용한 hyeshik.qbio.io의 mouselocalization 파일을 사용합니다.
# 인터넷 연결이 안 되면 이 셀은 실패할 수 있고, 그 경우 아래 localization 분석만 건너뛰면 됩니다.
import pandas as pd

local_url = 'https://hyeshik.qbio.io/binfo/mouselocalization-20210507.txt'
try:
    mouselocal = pd.read_csv(local_url, sep='\t')
    print(mouselocal.shape)
    display(mouselocal.head())
except Exception as e:
    mouselocal = None
    print('localization 파일을 불러오지 못했습니다:', e)


In [ ]:

if mouselocal is not None:
    # featureCounts 결과의 Geneid와 localization의 gene_id를 version 제거 후 merge합니다.
    tmp = df.reset_index().rename(columns={'index': 'Geneid'})
    tmp['gene_id_short'] = tmp['Geneid'].astype(str).str.split('.').str[0]
    mouselocal['gene_id_short'] = mouselocal['gene_id'].astype(str).str.split('.').str[0]

    merged = pd.merge(tmp, mouselocal, on='gene_id_short', how='inner')
    print('localization과 매칭된 유전자 수:', len(merged))
    display(merged.head())
else:
    merged = None


In [ ]:

if merged is not None and 'type' in merged.columns:
    # ER-associated translation과 관련 있을 가능성이 큰 category를 간단히 정의합니다.
    # 파일의 type 이름이 조금 달라도 membrane/ER/Golgi/secreted/extracellular 단어가 있으면 잡히도록 했습니다.
    pattern = r'membrane|endoplasmic|\bER\b|golgi|secret|extracellular|lumen'
    merged['er_membrane_related'] = merged['type'].astype(str).str.contains(pattern, case=False, regex=True)
    merged['local_group'] = np.where(merged['er_membrane_related'], 'ER/membrane/secretory-related', 'Other annotated')

    # 그룹별 비교
    a = merged.loc[merged['local_group'] == 'ER/membrane/secretory-related', 'rden_log2']
    b = merged.loc[merged['local_group'] == 'Other annotated', 'rden_log2']

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.boxplot([a, b], labels=['ER/membrane/secretory', 'Other'], showfliers=False)
    ax.axhline(0, linewidth=1)
    ax.set_ylabel('log2(ribosome density change)')
    ax.set_title('Translation change by protein localization')
    plt.tight_layout()
    plt.savefig('figures/fig3_localization_rden_boxplot.png', dpi=200)
    plt.show()

    stat, p = mannwhitneyu(a, b, alternative='two-sided')
    print(f'ER/membrane/secretory vs Other Mann-Whitney U p-value = {p:.3e}')
    print(f'ER/membrane/secretory median rden_log2 = {a.median():.4f}')
    print(f'Other median rden_log2 = {b.median():.4f}')

    # scatter plot에 localization 색을 입힙니다.
    fig, ax = plt.subplots(figsize=(6, 6))
    for label, sub in merged.groupby('local_group'):
        ax.scatter(sub['clip_log2'], sub['rden_log2'], s=8, alpha=0.35, label=label)
    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)
    ax.set_xlabel('log2(CLIP enrichment)')
    ax.set_ylabel('log2(ribosome density change)')
    ax.set_title('LIN28A target pattern by localization')
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.savefig('figures/fig4_localization_scatter.png', dpi=200)
    plt.show()
else:
    print('localization 분석을 건너뜁니다. mouselocal 파일 또는 type column이 없습니다.')


## 6. 후보 direct target gene table 만들기

In [ ]:

# 후보 direct target 선정
# 기준:
# 1) CLIP enrichment 상위 10%
# 2) Lin28a knockdown 후 ribosome density 증가: rden_log2 > 0
# 3) RNA abundance 자체 변화는 너무 크지 않음: abs(rna_change_log2) < 0.5
#    → 번역 변화에 좀 더 가까운 후보를 고르기 위한 조건입니다.

clip_cut = df['clip_log2'].quantile(0.90)

candidates = df[
    (df['clip_log2'] >= clip_cut) &
    (df['rden_log2'] > 0) &
    (df['rna_change_log2'].abs() < 0.5)
].copy()

# 점수: 결합 강도와 번역 증가를 동시에 반영
candidates['candidate_score'] = candidates['clip_log2'].rank(pct=True) + candidates['rden_log2'].rank(pct=True)
candidates = candidates.sort_values(['candidate_score', 'clip_log2', 'rden_log2'], ascending=False)

out_cols = [
    'Geneid', 'Chr', 'Start', 'End', 'Strand', 'Length',
    'CLIP-35L33G.bam', 'RNA-control.bam',
    'RPF-siLuc.bam', 'RNA-siLuc.bam',
    'RPF-siLin28a.bam', 'RNA-siLin28a.bam',
    'clip_log2', 'rden_log2', 'rna_change_log2', 'candidate_score'
]
out_cols = [c for c in out_cols if c in candidates.columns]

candidates.reset_index().rename(columns={'index': 'Geneid'})[out_cols].to_csv(
    'results/lin28a_candidate_direct_targets.csv', index=False
)

print('후보 direct target 수:', len(candidates))
display(candidates.reset_index().rename(columns={'index': 'Geneid'})[out_cols].head(20))


## 결과 해석 가이드

실행 후 다음처럼 해석하면 됩니다.

- `clip_log2`가 클수록 LIN28A가 해당 mRNA에 많이 결합한 것으로 해석합니다.
- `rden_log2 > 0`이면 Lin28a knockdown 후 ribosome density가 증가했다는 뜻입니다. 즉, 원래 LIN28A가 있을 때 번역이 억제되어 있었을 가능성이 있습니다.
- Top 5% binder의 `rden_log2` 중앙값이 Bottom 50%보다 크고 Mann-Whitney U test p-value가 작으면, **LIN28A 강결합 mRNA에서 knockdown 후 번역 증가가 더 뚜렷하다**고 정리할 수 있습니다.
- ER/membrane/secretory 관련 유전자군에서 `rden_log2`가 더 크면, 논문의 “LIN28A suppresses ER-associated translation” 결론과 같은 방향의 결과입니다.


In [ ]:

# 분석 결과 요약 파일 만들기
summary_lines = []
summary_lines.append('# LIN28A autonomous project summary')
summary_lines.append('')
summary_lines.append(f'- Number of genes used: {len(df)}')
summary_lines.append(f'- Spearman correlation between CLIP enrichment and ribosome density change: rho={rho:.4f}, p={pval:.3e}')
summary_lines.append(f'- Top 5% binder median rden_log2: {top.median():.4f}')
summary_lines.append(f'- Bottom 50% binder median rden_log2: {bottom.median():.4f}')
summary_lines.append(f'- Mann-Whitney U p-value, Top 5% vs Bottom 50%: {p:.3e}')
summary_lines.append(f'- Number of candidate direct targets: {len(candidates)}')
summary_lines.append('')
summary_lines.append('Generated files:')
summary_lines.append('- figures/fig1_clip_vs_rden_scatter.png')
summary_lines.append('- figures/fig2_rden_by_clip_group_boxplot.png')
summary_lines.append('- figures/fig3_localization_rden_boxplot.png, if localization data were available')
summary_lines.append('- figures/fig4_localization_scatter.png, if localization data were available')
summary_lines.append('- results/lin28a_candidate_direct_targets.csv')

with open('results/summary.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(summary_lines))

print('\n'.join(summary_lines))


## 7. GitHub에 push하기

In [ ]:

# GitHub push용 예시 코드입니다.
# 민감한 GitHub token을 노트북에 직접 적지 않는 것을 권합니다.
# Colab terminal이나 notebook에서 push할 때 GitHub가 비밀번호 대신 token 입력을 요구할 수 있습니다.

# 1) GitHub 웹사이트에서 빈 repository를 먼저 만드세요.
# 2) 아래 USER_NAME, REPO_NAME만 본인 정보로 바꾸세요.

USER_NAME = 'YOUR_GITHUB_USERNAME'
REPO_NAME = 'YOUR_REPOSITORY_NAME'

# 처음 올리는 경우
# !git clone https://github.com/{USER_NAME}/{REPO_NAME}.git
# !cp LIN28A_autonomous_project.ipynb {REPO_NAME}/
# !cp -r figures results {REPO_NAME}/
# %cd {REPO_NAME}
# !git add .
# !git commit -m "Add LIN28A autonomous project"
# !git push origin main

# 이미 repository 안에서 작업 중이면 아래만 실행해도 됩니다.
# !git add .
# !git commit -m "Update LIN28A autonomous project"
# !git push origin main
